<a href="https://colab.research.google.com/github/Chetalam/cnn-skin-cancer-classification/blob/main/notebooks/11_milk10k_clinical_data_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 11 — MILK10k clinical-data audit
Run on **CPU**. This notebook downloads three small official metadata CSVs and checks clinical SCC availability, diagnosis, skin-tone coverage and identifier overlap with the existing splits. It does not train, alter splits, replace a model, or download patient images.

Source: https://api.isic-archive.com/doi/milk10k/ (DOI 10.34970/648456). Attribution: MILK study team; CC-BY-NC. Keep attribution and licence with future image downloads.

MILK tone **0 = very dark, 5 = very light**. This is NOT Fitzpatrick I–VI. Never convert one scale to the other. Candidate rows are **not training-ready** until image-level duplicate checks and a prospective external split plan have been completed.

In [4]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from urllib.request import Request, urlopen
import hashlib
import json
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path('/content/drive/MyDrive/cnn-skin-cancer-classification')
STAMP = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f')
RAW_ROOT = PROJECT_ROOT / 'Datasets' / 'MILK10k' / 'metadata' / STAMP
OUTPUT_ROOT = PROJECT_ROOT / 'Reports' / 'MILK10k_Audit' / STAMP
RAW_ROOT.mkdir(parents=True, exist_ok=False)
OUTPUT_ROOT.mkdir(parents=True, exist_ok=False)
BASE = 'https://isic-archive.s3.amazonaws.com/dois/10.34970-648456/'
SOURCES = {'milk10k.csv': BASE + 'milk10k.csv',
           'training_supp.csv': BASE + 'supplements/training_supp.csv',
           'training_input.csv': BASE + 'supplements/training_input.csv'}
provenance = {}
for name, url in SOURCES.items():
    with urlopen(Request(url, headers={'User-Agent': 'AcademicDatasetAudit/1.0'}), timeout=120) as response:
        data = response.read(20_000_001)
    if len(data) > 20_000_000:
        raise ValueError('Metadata exceeds expected size; inspect the source before continuing.')
    path = RAW_ROOT / name
    path.write_bytes(data)
    provenance[name] = {'url': url, 'sha256': hashlib.sha256(data).hexdigest(), 'bytes': len(data)}
    print(name, len(data), 'bytes')
metadata = pd.read_csv(RAW_ROOT / 'milk10k.csv')
supplement = pd.read_csv(RAW_ROOT / 'training_supp.csv')
inputs = pd.read_csv(RAW_ROOT / 'training_input.csv')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
milk10k.csv 2129778 bytes
training_supp.csv 641153 bytes
training_input.csv 2587003 bytes


## Select diagnoses explicitly
Use invasive SCC only. Keratoacanthoma, actinic keratosis and SCC in situ are not silently mapped to invasive SCC. Melanoma includes invasive and in-situ diagnoses, excluding metastasis. Retain the detailed source diagnosis for review. Require histopathology for this candidate pool.

Image manipulation is reported, not assumed to mean a specific operation. Inspect the source documentation before deciding whether altered images belong in training.

In [5]:
def build_candidates(metadata, supplement, inputs):
    for name, frame in [('metadata', metadata), ('supplement', supplement), ('inputs', inputs)]:
        if frame['isic_id'].isna().any() or frame['isic_id'].duplicated().any():
            raise ValueError(name + ': missing or duplicate image identifiers')
    if set(metadata.isic_id) != set(supplement.isic_id) or set(metadata.isic_id) != set(inputs.isic_id):
        raise ValueError('Official metadata files contain different image identifiers.')
    source = metadata.merge(supplement[['isic_id', 'diagnosis_full']], on='isic_id', validate='one_to_one')
    source = source.merge(inputs[['isic_id', 'skin_tone_class']], on='isic_id', validate='one_to_one')
    tones = pd.to_numeric(source['skin_tone_class'], errors='raise')
    if not tones.dropna().isin(range(6)).all():
        raise ValueError('Unexpected MILK skin-tone category')
    source['skin_tone_class'] = tones.astype('Int64')
    labels = {'Basal cell carcinoma': 'BCC',
              'Melanoma Invasive': 'Melanoma',
              'Melanoma in situ': 'Melanoma',
              'Squamous cell carcinoma, Invasive': 'SCC'}
    source['class_label'] = source['diagnosis_full'].map(labels)
    clinical = source.loc[source['image_type'].eq('clinical: close-up')].copy()
    candidates = clinical.loc[clinical['class_label'].notna() & clinical['diagnosis_confirm_type'].eq('histopathology')].copy()
    if candidates.lesion_id.isna().any() or candidates.lesion_id.duplicated().any():
        raise ValueError('Expected one selected clinical photograph per recorded lesion.')
    candidates['dataset'] = 'MILK10k'
    candidates['tone_annotation_system'] = 'MILK_0_dark_to_5_light_NOT_Fitzpatrick'
    candidates['training_ready'] = False
    return source, clinical, candidates

source, clinical, candidates = build_candidates(metadata, supplement, inputs)
print('All clinical photographs:', len(clinical))
print('Histopathology-confirmed candidate photographs:', len(candidates))
display(candidates.groupby(['class_label', 'diagnosis_full']).size().rename('images').reset_index())
display(pd.crosstab(candidates['class_label'], candidates['skin_tone_class']).reindex(columns=range(6), fill_value=0))
print('Tone scale: 0 very dark → 5 very light; NOT Fitzpatrick.')
display(candidates.groupby(['class_label', 'image_manipulation'], dropna=False).size().rename('images').reset_index())

All clinical photographs: 5240
Histopathology-confirmed candidate photographs: 3267


,class_label,diagnosis_full,images
0,BCC,Basal cell carcinoma,2520
1,Melanoma,Melanoma Invasive,255
2,Melanoma,Melanoma in situ,181
3,SCC,"Squamous cell carcinoma, Invasive",311


skin_tone_class,0,1,2,3,4,5
class_label,,,,,,
BCC,1,19,77,1715,495,213
Melanoma,0,16,117,193,87,23
SCC,0,1,5,191,91,23


Tone scale: 0 very dark → 5 very light; NOT Fitzpatrick.


,class_label,image_manipulation,images
0,BCC,altered,91
1,BCC,instrument only,2429
2,Melanoma,altered,52
3,Melanoma,instrument only,384
4,SCC,altered,8
5,SCC,instrument only,303


## Exclude known overlapping lesions
Read identifiers only from the existing train, validation and test manifests. No test images, predictions or performance scores are loaded. If an original ISIC image is either member of a MILK pair, exclude both images of that MILK lesion.

These checks cannot detect unidentified duplicate or near-duplicate images. No patient identifier is supplied in these three files, so patient independence cannot be established from this audit.

In [6]:
known_ids = set()
split_hashes = {}
for split in ['train', 'validation', 'test']:
    path = PROJECT_ROOT / 'Datasets' / 'Splits' / f'{split}_metadata.csv'
    if not path.is_file():
        raise FileNotFoundError(f'Mount the correct Drive and locate this manifest: {path}')
    split_hashes[split] = hashlib.sha256(path.read_bytes()).hexdigest()
    existing = pd.read_csv(path, usecols=['image_id'], dtype='string')
    known_ids.update(existing.image_id.dropna().map(lambda value: Path(value).stem))

shared_lesions = set(source.loc[source.isic_id.isin(known_ids), 'lesion_id'].dropna())
candidates['known_original_lesion_overlap'] = candidates.lesion_id.isin(shared_lesions)
excluded = candidates.loc[candidates.known_original_lesion_overlap].copy()
pending = candidates.loc[~candidates.known_original_lesion_overlap].copy()
pending['eligibility_note'] = 'Pending byte/pixel/near-duplicate checks, prospective split and patient-overlap assessment'
print('Candidate clinical images excluded by known paired-lesion overlap:', len(excluded))
print('Candidates remaining for IMAGE-LEVEL audit:', len(pending))
display(pd.crosstab(pending.class_label, pending.skin_tone_class).reindex(columns=range(6), fill_value=0))

candidates.to_csv(OUTPUT_ROOT / 'all_histopathology_candidates.csv', index=False)
excluded.to_csv(OUTPUT_ROOT / 'known_overlap_exclusions.csv', index=False)
pending.to_csv(OUTPUT_ROOT / 'candidates_PENDING_image_checks.csv', index=False)
clinical.diagnosis_full.value_counts().rename_axis('diagnosis').reset_index(name='clinical_images').to_csv(OUTPUT_ROOT / 'all_clinical_diagnoses.csv', index=False)
pd.crosstab(pending.class_label, pending.skin_tone_class).reindex(columns=range(6), fill_value=0).to_csv(OUTPUT_ROOT / 'candidate_class_by_MILK_tone.csv')
report = {'source_doi': '10.34970/648456', 'attribution': 'MILK study team', 'licence': 'CC-BY-NC',
          'files': provenance, 'original_split_sha256': split_hashes,
          'histopathology_candidates': len(candidates), 'known_overlap_exclusions': len(excluded),
          'pending_candidates': len(pending), 'training_ready': False,
          'tone_scale': 'MILK 0 very dark to 5 very light; NOT Fitzpatrick',
          'no_patient_independence_claim': True, 'test_inference_performed': False}
(OUTPUT_ROOT / 'audit_provenance.json').write_text(json.dumps(report, indent=2), encoding='utf-8')
print('MILK10k METADATA AUDIT COMPLETE')
print('Reports saved to:', OUTPUT_ROOT)
print('No training, test inference or model replacement performed.')

Candidate clinical images excluded by known paired-lesion overlap: 0
Candidates remaining for IMAGE-LEVEL audit: 3267


skin_tone_class,0,1,2,3,4,5
class_label,,,,,,
BCC,1,19,77,1715,495,213
Melanoma,0,16,117,193,87,23
SCC,0,1,5,191,91,23


MILK10k METADATA AUDIT COMPLETE
Reports saved to: /content/drive/MyDrive/cnn-skin-cancer-classification/Reports/MILK10k_Audit/20261104_052201_662292
No training, test inference or model replacement performed.


## Interpretation and next experiment
This audit establishes available diagnoses and annotated tone counts, not model improvement. A large majority of MILK SCC cases may be in lighter categories; report the actual counts above. Adding MILK does not by itself establish darker-skin reliability.

Before training: download candidate clinical photographs with attribution; check original and external images for byte, decoded-pixel and near-duplicate overlap; link paired lesions; establish a fixed external split before fitting; and record remaining patient-identity limitations. Preserve the existing original splits.

Compare a real-only model against a separately documented real-plus-synthetic configuration on fixed real validation images. Synthetic data must be training-only, with generator-source leakage checked. Keep Fitzpatrick and MILK tone evaluations separate. Do not use synthetic test examples as evidence of real darker-skin performance.

Save this notebook to Drive and GitHub as `notebooks/11_milk10k_clinical_data_audit.ipynb`. Suggested commit message: `Record MILK10k clinical SCC availability and skin-tone audit`.